# 01 - SQL & PostgreSQL with Python
**Parts A-D:** relational databases, advanced SQL, PostgreSQL, SQLAlchemy connectivity.

Prerequisite: PostgreSQL is running, `.env` holds `DATABASE_URL`, and `python scripts/load_database.py` has been run.

In [1]:
import sys
sys.path.insert(0, "../scripts")          # reuse the project modules
import pandas as pd, numpy as np
from sqlalchemy import text
import db
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 30
engine = db.get_engine()

## 1. Connect to PostgreSQL

In [2]:
print(db.read_sql("SELECT version() AS postgres_version").iloc[0, 0])

PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## 2. Explore the schema (Part A: tables, keys, constraints, data types)

In [3]:
db.read_sql("""
SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'public' AND table_name IN
      ('regions','categories','customers','products','orders','order_items')
ORDER BY table_name, ordinal_position""")

,table_name,column_name,data_type,is_nullable
0,categories,category_id,smallint,NO
1,categories,category_name,character varying,NO
2,customers,customer_id,character varying,NO
3,customers,customer_name,character varying,NO
4,customers,region_id,smallint,NO
5,order_items,item_id,integer,NO
6,order_items,order_id,integer,NO
7,order_items,product_id,smallint,NO
8,order_items,quantity,integer,NO
9,order_items,unit_price,numeric,NO


In [4]:
# Keys and relationships
db.read_sql("""
SELECT tc.table_name, tc.constraint_type, tc.constraint_name
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'public' AND tc.constraint_type IN ('PRIMARY KEY','FOREIGN KEY','UNIQUE','CHECK')
  AND tc.constraint_name NOT LIKE '%not_null%'
ORDER BY tc.table_name, tc.constraint_type""")

,table_name,constraint_type,constraint_name
0,categories,PRIMARY KEY,categories_pkey
1,categories,UNIQUE,categories_category_name_key
2,customers,FOREIGN KEY,customers_region_id_fkey
3,customers,PRIMARY KEY,customers_pkey
4,order_items,CHECK,order_items_discount_check
5,order_items,CHECK,order_items_quantity_check
6,order_items,CHECK,order_items_unit_price_check
7,order_items,FOREIGN KEY,order_items_product_id_fkey
8,order_items,FOREIGN KEY,order_items_order_id_fkey
9,order_items,PRIMARY KEY,order_items_pkey


In [5]:
db.read_sql("""
SELECT 'regions' AS tbl, COUNT(*) AS rows FROM regions UNION ALL
SELECT 'categories', COUNT(*) FROM categories UNION ALL
SELECT 'customers', COUNT(*) FROM customers UNION ALL
SELECT 'products', COUNT(*) FROM products UNION ALL
SELECT 'orders', COUNT(*) FROM orders UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items""")

,tbl,rows
0,regions,5
1,categories,5
2,customers,292
3,products,20
4,orders,5825
5,order_items,5905


## 3. CRUD practice on a scratch table (CREATE / INSERT / SELECT / UPDATE / DELETE)

In [6]:
with engine.begin() as conn:                 # one transaction: all-or-nothing
    conn.execute(text("DROP TABLE IF EXISTS scratch_demo"))
    conn.execute(text("CREATE TABLE scratch_demo (id SERIAL PRIMARY KEY, name VARCHAR(40) NOT NULL, stock INT CHECK (stock >= 0))"))
    conn.execute(text("INSERT INTO scratch_demo (name, stock) VALUES (:n, :s)"),
                 [{"n": "Laptop", "s": 10}, {"n": "Tablet", "s": 25}, {"n": "Phone", "s": 40}])
print("After INSERT:\n", db.read_sql("SELECT * FROM scratch_demo ORDER BY id"))
with engine.begin() as conn:
    conn.execute(text("UPDATE scratch_demo SET stock = stock - 3 WHERE name = :n"), {"n": "Laptop"})
    conn.execute(text("DELETE FROM scratch_demo WHERE name = :n"), {"n": "Phone"})
print("\nAfter UPDATE + DELETE:\n", db.read_sql("SELECT * FROM scratch_demo ORDER BY id"))
# Constraint protects data: a negative stock must fail
try:
    with engine.begin() as conn:
        conn.execute(text("UPDATE scratch_demo SET stock = -5 WHERE name = 'Tablet'"))
except Exception as e:
    print("\nBlocked by CHECK constraint:", type(e).__name__)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE scratch_demo"))

After INSERT:
    id    name  stock
0   1  Laptop     10
1   2  Tablet     25
2   3   Phone     40

After UPDATE + DELETE:
    id    name  stock
0   1  Laptop      7
1   2  Tablet     25

Blocked by CHECK constraint: IntegrityError


## 4. Advanced SQL (Part B) - every query lives in `database/queries/`

In [7]:
def show(name, rows=12):
    sql = db.load_query(name)
    print(sql)
    return db.read_sql(sql).head(rows)

### SELECT, WHERE, ORDER BY, LIMIT

In [8]:
show("01_select_where_orderby")

-- 01 | SELECT, WHERE, ORDER BY, LIMIT
-- Question: What are the 10 biggest single order lines in the East region in 2025?
SELECT order_id, order_date, customer_name, product, quantity, revenue
FROM   vw_sales_detail
WHERE  region = 'East'
  AND  year   = 2025
ORDER  BY revenue DESC
LIMIT  10;



,order_id,order_date,customer_name,product,quantity,revenue
0,53822,2025-04-03,Meera Garcia,Laptop,3,"2,324.85"
1,50927,2025-09-02,Mia Khan,Laptop,3,"2,312.16"
2,51677,2025-01-04,Arjun Sharma,Laptop,3,"2,193.84"
3,51049,2025-04-08,Emma Wang,Laptop,3,"2,184.53"
4,50862,2025-05-24,Carlos Iyer,Laptop,3,"2,045.50"
5,51834,2025-12-24,Arjun Patel,Laptop,3,"2,020.71"
6,51937,2025-11-02,Arjun Silva,Laptop,3,"1,976.51"
7,51108,2025-12-31,Emma Wang,Laptop,3,"1,897.23"
8,52782,2025-06-05,Vikram Khan,Smartphone,3,"1,631.22"
9,51293,2025-06-21,Olivia Garcia,Smartphone,3,"1,584.03"


### GROUP BY + HAVING

In [9]:
show("02_group_by_having")

-- 02 | GROUP BY + HAVING + aggregate functions
-- Question: Which product categories have more than $100,000 in revenue, and what is their average order-line value?
SELECT category,
       COUNT(*)                    AS order_lines,
       SUM(quantity)               AS units_sold,
       ROUND(SUM(revenue), 2)      AS total_revenue,
       ROUND(AVG(revenue), 2)      AS avg_line_value
FROM   vw_sales_detail
GROUP  BY category
HAVING SUM(revenue) > 100000
ORDER  BY total_revenue DESC;



,category,order_lines,units_sold,total_revenue,avg_line_value
0,Technology,1499,2264,"811,629.81",541.45
1,Furniture,837,1063,"340,622.29",406.96
2,Clothing,1176,2379,"145,815.89",123.99
3,Home & Kitchen,861,1299,"133,491.55",155.04


### JOINS

In [10]:
show("03_joins", 8)

-- 03 | INNER JOIN, LEFT JOIN across the schema (not the view, to show the joins explicitly)
-- Question: Revenue per region and category, plus products that have never been sold (LEFT JOIN + IS NULL).
SELECT r.region_name,
       cat.category_name,
       COUNT(oi.item_id)              AS order_lines,
       ROUND(SUM(oi.line_revenue), 2) AS revenue
FROM   order_items oi
INNER JOIN orders     o   ON o.order_id      = oi.order_id
INNER JOIN customers  c   ON c.customer_id   = o.customer_id
INNER JOIN regions    r   ON r.region_id     = c.region_id
INNER JOIN products   p   ON p.product_id    = oi.product_id
INNER JOIN categories cat ON cat.category_id = p.category_id
GROUP  BY r.region_name, cat.category_name
ORDER  BY r.region_name, revenue DESC;

-- Products that have never been sold (empty result = every product sells)
-- SELECT p.product_name
-- FROM   products p
-- LEFT JOIN order_items oi ON oi.product_id = p.product_id
-- WHERE  oi.item_id IS NULL;



,region_name,category_name,order_lines,revenue
0,Central,Technology,134,"74,708.81"
1,Central,Furniture,91,"34,418.93"
2,Central,Clothing,97,"11,954.63"
3,Central,Home & Kitchen,61,"10,244.33"
4,Central,Office Supplies,113,"6,392.95"
5,East,Technology,544,"299,991.45"
6,East,Furniture,301,"116,328.90"
7,East,Clothing,443,"53,098.76"


### Subquery

In [11]:
show("04_subquery_above_average", 8)

-- 04 | Subqueries
-- Question: Which customers spend more than the average customer? (scalar subquery in HAVING)
SELECT customer_id, customer_name, region, orders, ROUND(revenue, 2) AS revenue
FROM   vw_customer_summary
WHERE  revenue > (SELECT AVG(revenue) FROM vw_customer_summary)
ORDER  BY revenue DESC
LIMIT  15;



,customer_id,customer_name,region,orders,revenue
0,CUST-1269,Liam Sharma,East,262,"65,795.85"
1,CUST-1054,Emma Wang,East,162,"48,958.85"
2,CUST-1099,Arjun Silva,East,123,"41,390.61"
3,CUST-1065,Olivia Garcia,East,104,"29,109.70"
4,CUST-1008,Liam Patel,South,85,"26,093.40"
5,CUST-1073,Aarav Reddy,South,80,"25,531.96"
6,CUST-1176,Sneha Smith,East,93,"23,739.02"
7,CUST-1169,Vikram Silva,East,78,"22,552.15"


### CTE + LAG (month-over-month growth)

In [12]:
show("05_cte_monthly_growth", 6)

-- 05 | CTE + LAG window function
-- Question: Which months grew the most vs the previous month, and which fell the most?
WITH monthly AS (
    SELECT year_month, SUM(revenue) AS revenue
    FROM   vw_sales_detail
    GROUP  BY year_month
), growth AS (
    SELECT year_month,
           ROUND(revenue, 2) AS revenue,
           ROUND((revenue / LAG(revenue) OVER (ORDER BY year_month) - 1) * 100, 1) AS mom_growth_pct
    FROM   monthly
)
SELECT *
FROM   growth
WHERE  mom_growth_pct IS NOT NULL
ORDER  BY mom_growth_pct DESC;



,year_month,revenue,mom_growth_pct
0,2024-02,"49,730.78",43.10
1,2024-12,"101,432.41",41.90
2,2024-05,"66,230.89",30.80
3,2025-11,"95,929.95",29.00
4,2025-10,"74,336.88",28.50
5,2025-03,"59,676.51",16.10


### Window functions: running total and ranking

In [13]:
show("06_window_running_total", 14)

-- 06 | Window functions: running total, share of year, ranking inside each year
-- Question: How does revenue accumulate through each year, and which months rank highest within their year?
WITH monthly AS (
    SELECT year, month, SUM(revenue) AS revenue
    FROM   vw_sales_detail
    GROUP  BY year, month
)
SELECT year, month,
       ROUND(revenue, 2)                                                  AS revenue,
       ROUND(SUM(revenue) OVER (PARTITION BY year ORDER BY month), 2)     AS running_total,
       ROUND(revenue * 100.0 / SUM(revenue) OVER (PARTITION BY year), 1)  AS pct_of_year,
       RANK() OVER (PARTITION BY year ORDER BY revenue DESC)              AS rank_in_year
FROM   monthly
ORDER  BY year, month;



,year,month,revenue,running_total,pct_of_year,rank_in_year
0,2024,1,"34,748.70","34,748.70",4.70,12
1,2024,2,"49,730.78","84,479.48",6.70,11
2,2024,3,"53,099.19","137,578.67",7.10,9
3,2024,4,"50,618.52","188,197.19",6.80,10
4,2024,5,"66,230.89","254,428.08",8.90,4
5,2024,6,"62,976.27","317,404.35",8.50,6
6,2024,7,"56,035.14","373,439.49",7.50,8
7,2024,8,"59,477.13","432,916.62",8.00,7
8,2024,9,"64,361.60","497,278.22",8.70,5
9,2024,10,"72,938.54","570,216.76",9.80,2


### Top N per group (ROW_NUMBER / RANK with PARTITION BY)

In [14]:
show("07_top_products_per_category")

-- 07 | ROW_NUMBER() with PARTITION BY: "top N per group"
-- Question: What are the top 2 products in every category?
SELECT category, product, ROUND(revenue, 2) AS revenue, category_rank
FROM   vw_product_performance
WHERE  category_rank <= 2
ORDER  BY category, category_rank;



,category,product,revenue,category_rank
0,Clothing,Jacket,"52,279.25",1
1,Clothing,Sneakers,"48,154.30",2
2,Furniture,Sofa,"156,273.45",1
3,Furniture,Standing Desk,"103,949.31",2
4,Home & Kitchen,Cookware Set,"56,619.62",1
5,Home & Kitchen,Air Fryer,"46,806.10",2
6,Office Supplies,Printer Paper,"37,036.05",1
7,Office Supplies,Stapler,"20,641.54",2
8,Technology,Laptop,"300,998.79",1
9,Technology,Smartphone,"241,829.49",2


### RFM segmentation in SQL (CTE + NTILE)

In [15]:
show("08_customer_rfm", 10)

-- 08 | RFM customer segmentation in pure SQL (CTE + NTILE)
-- Recency = days since last order, Frequency = number of orders, Monetary = total revenue.
-- NTILE(5) splits customers into 5 equal groups; 5 = best.
WITH snapshot AS (
    SELECT MAX(order_date) + 1 AS snap FROM orders
), rfm AS (
    SELECT cs.customer_id, cs.customer_name, cs.region,
           (s.snap - cs.last_order) AS recency_days,
           cs.orders                AS frequency,
           cs.revenue               AS monetary
    FROM   vw_customer_summary cs CROSS JOIN snapshot s
    WHERE  cs.orders > 0
), scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,   -- fewer days = higher score
           NTILE(5) OVER (ORDER BY frequency)         AS f_score,
           NTILE(5) OVER (ORDER BY monetary)          AS m_score
    FROM rfm
)
SELECT customer_id, customer_name, recency_days, frequency, ROUND(monetary, 2) AS monetary,
       r_score, f_score, m_score,
       CASE WHEN r_s

,customer_id,customer_name,recency_days,frequency,monetary,r_score,f_score,m_score,segment
0,CUST-1269,Liam Sharma,1,262,"65,795.85",5,5,5,Champions
1,CUST-1054,Emma Wang,1,162,"48,958.85",5,5,5,Champions
2,CUST-1099,Arjun Silva,2,123,"41,390.61",5,5,5,Champions
3,CUST-1065,Olivia Garcia,2,104,"29,109.70",5,5,5,Champions
4,CUST-1008,Liam Patel,11,85,"26,093.40",4,5,5,Champions
5,CUST-1073,Aarav Reddy,6,80,"25,531.96",5,5,5,Champions
6,CUST-1176,Sneha Smith,4,93,"23,739.02",5,5,5,Champions
7,CUST-1169,Vikram Silva,3,78,"22,552.15",5,5,5,Champions
8,CUST-1263,Chen Iyer,4,84,"21,396.38",5,5,5,Champions
9,CUST-1193,Priya Muller,1,68,"21,032.52",5,5,5,Champions


### Pareto (cumulative share)

In [16]:
p = db.run_query("09_pareto_customers"); print(f"{len(p)} customers (out of 292) generate 50% of revenue"); p.tail(3)

40 customers (out of 292) generate 50% of revenue


,rnk,customer_name,revenue,cumulative_pct
37,38,Mia Iyer,"10,359.83",48.20
38,39,Anita Singh,"9,923.75",48.90
39,40,Emma Muller,"9,759.04",49.50


### Conditional aggregation with FILTER, and UNION ALL

In [17]:
show("10_yoy_category_filter")

-- 10 | Conditional aggregation with FILTER (pivot in SQL)
-- Question: Revenue per category for 2024 vs 2025 and the growth between them.
SELECT category,
       ROUND(SUM(revenue) FILTER (WHERE year = 2024), 2) AS revenue_2024,
       ROUND(SUM(revenue) FILTER (WHERE year = 2025), 2) AS revenue_2025,
       ROUND((SUM(revenue) FILTER (WHERE year = 2025) /
              SUM(revenue) FILTER (WHERE year = 2024) - 1) * 100, 1) AS growth_pct
FROM   vw_sales_detail
GROUP  BY category
ORDER  BY growth_pct;



,category,revenue_2024,revenue_2025,growth_pct
0,Home & Kitchen,"67,841.19","65,650.36",-3.20
1,Furniture,"167,859.78","172,762.51",2.90
2,Technology,"394,951.10","416,678.71",5.50
3,Office Supplies,"42,916.42","45,788.02",6.70
4,Clothing,"69,579.82","76,236.07",9.60


In [18]:
show("11_union_top_entities")

-- 11 | UNION ALL: combine result sets of different entities
-- Question: A single leaderboard of the top 3 regions, top 3 categories and top 3 products.
(SELECT 'Region' AS entity, region AS name, ROUND(SUM(revenue), 2) AS revenue
 FROM vw_sales_detail GROUP BY region ORDER BY revenue DESC LIMIT 3)
UNION ALL
(SELECT 'Category', category, ROUND(SUM(revenue), 2)
 FROM vw_sales_detail GROUP BY category ORDER BY 3 DESC LIMIT 3)
UNION ALL
(SELECT 'Product', product, ROUND(SUM(revenue), 2)
 FROM vw_sales_detail GROUP BY product ORDER BY 3 DESC LIMIT 3);



,entity,name,revenue
0,Region,East,"552,621.13"
1,Region,South,"292,122.32"
2,Region,West,"287,247.62"
3,Category,Technology,"811,629.81"
4,Category,Furniture,"340,622.29"
5,Category,Clothing,"145,815.89"
6,Product,Laptop,"300,998.79"
7,Product,Smartphone,"241,829.49"
8,Product,Sofa,"156,273.45"


## 5. Views (Part B)
Views are saved queries. The dashboard reads from them.

In [19]:
db.read_sql("SELECT viewname FROM pg_views WHERE schemaname='public' ORDER BY 1")

,viewname
0,vw_category_yearly
1,vw_customer_summary
2,vw_monthly_revenue
3,vw_product_performance
4,vw_sales_detail


In [20]:
db.read_sql("SELECT * FROM vw_product_performance ORDER BY overall_rank LIMIT 6")

,product,category,units,revenue,revenue_share_pct,overall_rank,category_rank
0,Laptop,Technology,418,"300,998.79",19.80,1,1
1,Smartphone,Technology,490,"241,829.49",15.91,2,2
2,Sofa,Furniture,257,"156,273.45",10.28,3,1
3,Tablet,Technology,446,"140,712.96",9.26,4,3
4,Standing Desk,Furniture,262,"103,949.31",6.84,5,2
5,Smartwatch,Technology,442,"87,931.95",5.78,6,4


## 6. Query optimisation: EXPLAIN ANALYZE and indexes

In [21]:
plan = db.run_query("12_explain_index_demo")
print("\n".join(plan["QUERY PLAN"]))

Hash Join  (cost=17.14..147.70 rows=266 width=14) (actual time=0.126..1.463 rows=289 loops=1)
  Hash Cond: (oi.order_id = o.order_id)
  ->  Seq Scan on order_items oi  (cost=0.00..115.05 rows=5905 width=10) (actual time=0.006..0.550 rows=5905 loops=1)
  ->  Hash  (cost=13.87..13.87 rows=262 width=8) (actual time=0.112..0.113 rows=262 loops=1)
        Buckets: 1024  Batches: 1  Memory Usage: 19kB
        ->  Index Scan using idx_orders_customer on orders o  (cost=0.28..13.87 rows=262 width=8) (actual time=0.013..0.069 rows=262 loops=1)
              Index Cond: ((customer_id)::text = 'CUST-1269'::text)
Planning Time: 0.282 ms
Execution Time: 1.510 ms


In [22]:
# Compare: drop the index, look at the plan, then recreate it
with engine.begin() as conn:
    conn.execute(text("DROP INDEX idx_orders_customer"))
def plan_for(sql):
    return "\n".join(db.read_sql("EXPLAIN ANALYZE " + sql)["QUERY PLAN"])
q = "SELECT * FROM orders WHERE customer_id = 'CUST-1269'"
print("WITHOUT index:\n", plan_for(q).split("\n")[0], "\n", plan_for(q).split("\n")[-2:])
with engine.begin() as conn:
    conn.execute(text("CREATE INDEX idx_orders_customer ON orders (customer_id)"))
print("\nWITH index:\n", plan_for(q).split("\n")[0], "\n", plan_for(q).split("\n")[-2:])

WITHOUT index:
 Seq Scan on orders  (cost=0.00..115.81 rows=262 width=27) (actual time=0.448..0.547 rows=262 loops=1) 
 ['Planning Time: 0.054 ms', 'Execution Time: 0.578 ms']

WITH index:
 Index Scan using idx_orders_customer on orders  (cost=0.28..13.87 rows=262 width=27) (actual time=0.012..0.050 rows=262 loops=1) 
 ['Planning Time: 0.056 ms', 'Execution Time: 0.093 ms']


## 7. Python <-> SQL (Part D): read, write and safe parameters

In [23]:
# Reading SQL into pandas with a bound parameter (safe)
df = db.read_sql("SELECT region, ROUND(SUM(revenue),2) AS revenue FROM vw_sales_detail WHERE year = :y GROUP BY region ORDER BY 2 DESC", {"y": 2025})
df

,region,revenue
0,East,"284,559.23"
1,West,"149,487.65"
2,South,"144,536.94"
3,North,"130,801.72"
4,Central,"67,730.13"


In [24]:
# Writing a DataFrame to SQL, reading it back, cleaning up
df.to_sql("tmp_region_revenue", engine, if_exists="replace", index=False)
print(db.read_sql("SELECT * FROM tmp_region_revenue"))
with engine.begin() as conn:
    conn.execute(text("DROP TABLE tmp_region_revenue"))

    region    revenue
0     East 284,559.23
1     West 149,487.65
2    South 144,536.94
3    North 130,801.72
4  Central  67,730.13


In [25]:
# SQL injection attempt: bound parameters treat it as plain text
evil = "x'; DROP TABLE orders; --"
r = db.read_sql("SELECT COUNT(*) AS n FROM vw_sales_detail WHERE customer_name = :name", {"name": evil})
print("Matches:", r.iloc[0, 0], "| orders table still has", db.read_sql("SELECT COUNT(*) FROM orders").iloc[0, 0], "rows")

Matches: 0 | orders table still has 5825 rows


## Key takeaways
- Normalised tables + keys + constraints protect data quality; `line_revenue` is a **generated column**, so revenue can never drift.
- CTEs and window functions replace slow, repetitive Python loops.
- Always pass values as bound parameters. Never build SQL with f-strings from user input.